In [7]:
import pandas as pd
import os

# --- CONFIGURAZIONE ---
PATH_INPUT = 'data/train/training_data.csv'
DIR_OUTPUT = 'data_elaborated/train/'
FILE_OUTPUT = 'train_cleaned.csv'

def surgical_clean(df, is_training=True):
    print("Inizio pulizia...")
    
    # 1. Copia e pulizia base
    df_clean = df.copy()
    sensori_nomi = [c for c in df_clean.columns if 'Sensed' in c]
    
    # Rimuoviamo duplicati e filtriamo altitudine
    df_clean = df_clean.drop_duplicates(subset=['ESN', 'Cycles_Since_New', 'Snapshot'] + sensori_nomi)
    if 'Sensed_Altitude' in df_clean.columns:
        df_clean = df_clean[df_clean['Sensed_Altitude'] >= 0]
    
    # Bias 20k
    if is_training:
        df_clean = df_clean[df_clean['Cycles_Since_New'] < 20000]

    # 2. Clipping degli Outlier (Metodo Sicuro)
    print("Esecuzione Clipping Outlier per motore e fase di volo...")
    
    cleaned_chunks = []
    # Raggruppiamo per ESN e Snapshot
    grouped = df_clean.groupby(['ESN', 'Snapshot'])
    
    for (esn, snapshot), group in grouped:
        group = group.copy()
        for s in sensori_nomi:
            # Calcolo IQR
            Q1 = group[s].quantile(0.25)
            Q3 = group[s].quantile(0.75)
            IQR = Q3 - Q1
            # Applichiamo il clipping
            group[s] = group[s].clip(lower=Q1 - 1.5 * IQR, upper=Q3 + 1.5 * IQR)
        
        cleaned_chunks.append(group)

    # 3. Ricostruzione finale
    df_final = pd.concat(cleaned_chunks, ignore_index=True)
    
    # Rimuoviamo eventuali colonne "index" o "level_0" create da pandas
    df_final = df_final.loc[:, ~df_final.columns.str.contains('^Unnamed|^index|^level_0')]
    
    return df_final

if __name__ == "__main__":
    if os.path.exists(PATH_INPUT):
        print(f"Caricamento {PATH_INPUT}...")
        df_raw = pd.read_csv(PATH_INPUT)
        
        # Esecuzione
        df_cleaned = surgical_clean(df_raw, is_training=True)
        
        # VERIFICA FINALE
        if 'ESN' in df_cleaned.columns:
            if not os.path.exists(DIR_OUTPUT): os.makedirs(DIR_OUTPUT)
            df_cleaned.to_csv(os.path.join(DIR_OUTPUT, FILE_OUTPUT), index=False)
            print("-" * 30)
            print(f"SUCCESSO TOTALE!")
            print(f"File salvato in: {os.path.join(DIR_OUTPUT, FILE_OUTPUT)}")
            print(f"Colonne salvate: {df_cleaned.columns.tolist()[:8]}...")
        else:
            print("\n!!! ERRORE CRITICO: ESN non trovato neanche con il metodo Chunk !!!")
            print(f"Colonne attuali: {df_cleaned.columns.tolist()}")
    else:
        print(f"ERRORE: File {PATH_INPUT} non trovato.")

Caricamento data/train/training_data.csv...
Inizio pulizia...
Esecuzione Clipping Outlier per motore e fase di volo...
------------------------------
SUCCESSO TOTALE!
File salvato in: data_elaborated/train/train_cleaned.csv
Colonne salvate: ['ESN', 'Cycles_Since_New', 'Snapshot', 'Cumulative_WWs', 'Cumulative_HPC_SVs', 'Cumulative_HPT_SVs', 'Sensed_Altitude', 'Sensed_Mach']...
